In [3]:
import json
import ssl
import urllib.parse
import urllib.request

import certifi

params = urllib.parse.urlencode(
    {
        "start": "1",
        "limit": "15",
        "convert": "USD",
    }
)

request = urllib.request.Request(
    f"https://pro-api.coinmarketcap.com/v1/cryptocurrency/listings/latest?{params}",
    headers={
        "Accept": "application/json",
        "X-CMC_PRO_API_KEY": "44a36217e2644c90ae7e96136fb4ea57",
    },
)

context = ssl.create_default_context(cafile=certifi.where())

with urllib.request.urlopen(request, context=context) as response:
    data = json.load(response)

print(data)


{'data': [{'id': 1, 'name': 'Bitcoin', 'symbol': 'BTC', 'slug': 'bitcoin', 'infinite_supply': False, 'circulating_supply': 20022387, 'total_supply': 20022387, 'max_supply': 21000000, 'date_added': '2010-07-13T00:00:00.000Z', 'num_market_pairs': 12621, 'cmc_rank': 1, 'last_updated': '2026-04-29T18:01:00.000Z', 'tvl_ratio': None, 'platform': None, 'self_reported_circulating_supply': None, 'self_reported_market_cap': None, 'minted_market_cap': 1523355903403.02, 'quote': {'USD': {'price': 76082.63207593668, 'volume_24h': 36971643189.94605, 'cex_volume_24h': 36926270029.02831, 'dex_volume_24h': 45373160.9177476, 'volume_change_24h': 11.2113, 'percent_change_1h': 0.34983863, 'percent_change_24h': -0.09318046, 'percent_change_7d': -3.66952201, 'percent_change_30d': 13.94395425, 'percent_change_60d': 16.41401031, 'percent_change_90d': -9.60548965, 'market_cap': 1523355903403.0176, 'market_cap_dominance': 60.0016, 'fully_diluted_market_cap': 1597735273594.67, 'tvl': None, 'last_updated': '2026-

In [4]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
df = pd.json_normalize(data['data'])
df['timestamp'] = pd.to_datetime('now')


def api_runner():
    params = urllib.parse.urlencode(
        {
            "start": "1",
            "limit": "15",
            "convert": "USD",
        }
    )

    request = urllib.request.Request(
        f"https://pro-api.coinmarketcap.com/v1/cryptocurrency/listings/latest?{params}",
        headers={
            "Accept": "application/json",
            "X-CMC_PRO_API_KEY": "44a36217e2644c90ae7e96136fb4ea57",
        },
    )

    context = ssl.create_default_context(cafile=certifi.where())

    with urllib.request.urlopen(request, context=context) as response:
        data = json.load(response)

    #normalize API Data
    df2 = pd.json_normalize(data['data'])

    #Add timestamp
    df2['timestamp'] = pd.Timestamp.now()


    # keep only useful columns
    df2 = df2[
        [
            "id",
            "name",
            "symbol",
            "cmc_rank",
            "quote.USD.price",
            "quote.USD.volume_24h",
            "quote.USD.percent_change_1h",
            "quote.USD.percent_change_24h",
            "quote.USD.percent_change_7d",
            "quote.USD.market_cap",
            "timestamp",
        ]
    ]

    #remane columns
    df2 = df2.rename(columns={
        "quote.USD.price": "price_usd",
        "quote.USD.volume_24h": "volume_24h",
        "quote.USD.percent_change_1h": "pct_change_1h",
        "quote.USD.percent_change_24h": "pct_change_24h",
        "quote.USD.percent_change_7d": "pct_change_7d",
        "quote.USD.market_cap": "market_cap"
    })



    #Convert numeric columns for standardization
    numeric_cols = [
        "price_usd",
        "volume_24h",
        "pct_change_1h",
        "pct_change_24h",
        "pct_change_7d",
        "market_cap"
    ]

    df2[numeric_cols] = df2[numeric_cols].apply(pd.to_numeric, errors='coerce')

    #Remove duplicates from the API pull
    df2 = df2.drop_duplicates(subset=["id", "timestamp"])

    #sort by rank
    df2 = df2.sort_values("cmc_rank")


    return df2

import os
from time import time
from time import sleep

df = pd.DataFrame()
file_path = "crypto_historical_data.csv"


for i in range(333):
        df2 = api_runner()

        df = pd.concat([df, df2], ignore_index=True) #memory testing

        # save to CVS (append logic)
        write_header = not os.path.exists(file_path)

        df2.to_csv(file_path, mode="a", header=write_header, index=False)

        print('Historical data updated successfully')
        sleep(60) #sleep for 1 minute


Historical data updated successfully
Historical data updated successfully
Historical data updated successfully
Historical data updated successfully
Historical data updated successfully
Historical data updated successfully
Historical data updated successfully
Historical data updated successfully
Historical data updated successfully
Historical data updated successfully
Historical data updated successfully
Historical data updated successfully
Historical data updated successfully
Historical data updated successfully
Historical data updated successfully
Historical data updated successfully
Historical data updated successfully
Historical data updated successfully
Historical data updated successfully
Historical data updated successfully
Historical data updated successfully
Historical data updated successfully
Historical data updated successfully
Historical data updated successfully
Historical data updated successfully
Historical data updated successfully
Historical data updated successfully
H

KeyboardInterrupt: 